In [28]:
# #Uncomment these in a fresh notebook environment.
# %cd /content/marl-tsc
# %pip install -r requirements.txt
# %pip install -e .

In [29]:
#GITHUB --> COLAB CLONE / env setup
import os
from google.colab import userdata

# 1. Get your access token
token = userdata.get('GH_TOKEN')

# 2. Clone the repo if it's not already there
repo_path = '/content/marl-tsc'
if not os.path.exists(repo_path):
    !git clone --branch tom_new --single-branch https://{token}@github.com/abergh18/marl-tsc.git

# 3. CRUCIAL: Move to the root directory
# This ensures that 'import models' or 'import utils' works correctly
%cd {repo_path}

# 4. Install dependencies
!pip install -r requirements.txt
%pip install -e .

/content/marl-tsc
  Using cached traci-1.26.0-py2.py3-none-any.whl.metadata (3.2 kB)
  Using cached sumolib-1.26.0-py2.py3-none-any.whl.metadata (2.6 kB)
Using cached traci-1.26.0-py2.py3-none-any.whl (138 kB)
Using cached sumolib-1.26.0-py2.py3-none-any.whl (233 kB)
  Attempting uninstall: sumolib
    Found existing installation: sumolib 1.27.1
    Uninstalling sumolib-1.27.1:
      Successfully uninstalled sumolib-1.27.1
  Attempting uninstall: traci
    Found existing installation: traci 1.27.1
    Uninstalling traci-1.27.1:
      Successfully uninstalled traci-1.27.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libsumo 1.27.1 requires traci>=1.27.1, but you have traci 1.26.0 which is incompatible.
Obtaining file:///content/marl-tsc
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements

In [30]:
!pip install eclipse-sumo libsumo

import os
os.environ["SUMO_HOME"] = "/usr/local/lib/python3.12/dist-packages/sumo"
os.environ["PATH"] += ":/usr/local/lib/python3.12/dist-packages/sumo/bin"

  Using cached traci-1.27.1-py2.py3-none-any.whl.metadata (3.2 kB)
  Using cached sumolib-1.27.1-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached traci-1.27.1-py2.py3-none-any.whl (138 kB)
Using cached sumolib-1.27.1-py2.py3-none-any.whl (236 kB)
  Attempting uninstall: sumolib
    Found existing installation: sumolib 1.26.0
    Uninstalling sumolib-1.26.0:
      Successfully uninstalled sumolib-1.26.0
  Attempting uninstall: traci
    Found existing installation: traci 1.26.0
    Uninstalling traci-1.26.0:
      Successfully uninstalled traci-1.26.0


In [31]:
from pathlib import Path
import sys
#Had to remove '.parent' for colab specific version
PROJECT_ROOT = Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [32]:
from pathlib import Path

import matplotlib.pyplot as plt

GREEN_PHASE_COUNT = 4

def print_comparison_table(rows):
    columns = [
        "Policy",
        "Episodes",
        "Mean total reward",
        "Mean local queue",
        "Total completed steps",
        "Mean switches / episode",
    ]

    def format_value(value):
        if isinstance(value, float):
            return f"{value:.3f}"
        return str(value)

    widths = {column: len(column) for column in columns}
    for row in rows:
        for column in columns:
            widths[column] = max(widths[column], len(format_value(row.get(column, ""))))

    header = " | ".join(column.ljust(widths[column]) for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(separator)
    for row in rows:
        print(" | ".join(format_value(row.get(column, "")).ljust(widths[column]) for column in columns))


In [33]:
from marl_tsc.simulation_generator import SimulationGenerator, DEFAULT_TRAFFIC_LIGHT_IDS
from marl_tsc.training import train_ppo, evaluate_policy, plot_training_histories
from marl_tsc.mappo import train_mappo
from marl_tsc.baselines import random_actions, fixed_time_actions

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "main_flow.ipynb").exists() else NOTEBOOK_DIR
OUTPUT_DIR = PROJECT_ROOT / "outputs"

traffic_light_ids = list(DEFAULT_TRAFFIC_LIGHT_IDS)


## 1. Generate the SUMO simulation files

This default setup is a small `4x4` grid with four traffic-light agents. Change these values first when you want a different learning problem.


In [51]:
import shutil
from pathlib import Path

# ==========================================
# MASTER VARIABLE: Controls the entire run
# ==========================================
EPISODE_STEPS = 720  # The number of actions/steps the RL agent takes per episode


# ==========================================
# Derived Parameters
# ==========================================
SECONDS_PER_ACTION = 5  # Matches the step logic in your SumoTrafficEnv
# The actual duration in SUMO seconds that the RL agent will experience
SIMULATION_DURATION = EPISODE_STEPS * SECONDS_PER_ACTION  # 3000 SUMO seconds

# To ensure continuous flow, spawn vehicles for 20% longer than the simulation runs.
# This guarantees that the network never empties out or hits ACT = 0.
TRAFFIC_SPAWN_DURATION = int(SIMULATION_DURATION * 1.20)  # 3600 SUMO seconds

NUM_EPISODES = 30
EVALUATION_EPISODES = 10
TOTAL_TIMESTEPS = SIMULATION_DURATION
MIN_GREEN_SECONDS = 10


# ==========================================
# Generator Configuration
# ==========================================

# Ensure the output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The name of your uploaded network file
network_file_name = "lancaster_373m_35.net.xml"
osm_file_name = "lancaster_373m_35.osm.xml"

net_path_in_output = OUTPUT_DIR / network_file_name
osm_path_in_repo = Path.cwd() / osm_file_name

if net_path_in_output.exists():
    os.remove(net_path_in_output)

if not net_path_in_output.exists() and osm_path_in_repo.exists():
    print(f"Found OSM network file '{osm_file_name}', converting to '{network_file_name}'...")
    # It's important that OUTPUT_DIR exists before this command
    !netconvert --osm-files {osm_path_in_repo} --output-file {net_path_in_output} \
        --tls.guess true \
        --tls.join true \
        --tls.guess-signals true \
        --junctions.join true \
        --xml-validation never
    if net_path_in_output.exists(): # Only print success if the file actually exists after conversion
        print(f"Conversion complete. Network file '{network_file_name}' created at {net_path_in_output}")
    else:
        print(f"Conversion failed. Network file '{network_file_name}' was not created at {net_path_in_output}")

# Potential source paths for the network file
potential_source_paths = [
    net_path_in_output, # Prioritize the converted file in output directory
    Path.cwd() / network_file_name,  # In the project root (/content/marl-tsc/)
    Path('/content/') / network_file_name, # In the Colab root (/content/)
]

found_source_path = None
for p in potential_source_paths:
    if p.exists():
        found_source_path = p
        break

destination_path = OUTPUT_DIR / network_file_name

if found_source_path is None:
    print(f"ERROR: Network file '{network_file_name}' not found in any expected location: {Path.cwd()}, /content/, or {OUTPUT_DIR}.")
    print("Please ensure the network file is uploaded to one of these locations or an '.osm.xml' counterpart is present for conversion.")
    raise FileNotFoundError(f"Network file '{network_file_name}' is missing.")
elif found_source_path != destination_path:
    shutil.move(found_source_path, destination_path)
    print(f"Moved network file from {found_source_path} to {destination_path}")
else:
    print(f"Network file '{network_file_name}' already exists at {destination_path}. No move needed.")


# Paste the IDs you extracted in Step 1 here
custom_tls_ids = [
  '5464450922',
  'cluster_1111602959_6308330_6398665769_6398666101',
  'cluster_1112902336_2338735_6300917286',
  'cluster_1395459938_3904465077_6288784963',
  'cluster_1671362532_2217405_253943174_3904937313_#3more',
  'cluster_2217373_5464742560_610595545_6288784968_#6more',
  'cluster_27212021_5464547758_6299153326',
  'joinedS_3903073405_cluster_5835265011_610595547',
  'joinedS_6528610881_cluster_1305014216_5464547754_6100910_6528610880_cluster_2217469_5464547745_5464547748_5464547749',
  'joinedS_cluster_1670999797_1670999973_2217366_3903086088_#1more_cluster_1670999799_2217365_6308327',
  'joinedS_cluster_3903235205_3903235208_6288785520_cluster_3903235206_3903235211_3903235213_6288785518_#1more_cluster_5178274273_6288785516'
]

# Configure the generator to use your existing network
generator = SimulationGenerator(
    output_dir=OUTPUT_DIR,
    network_filename=network_file_name, # Point to your file (now in OUTPUT_DIR)
    traffic_light_ids=custom_tls_ids,
    # You don't need grid_number or lane_number for real-world maps
)

# Crucial: Instead of generate_all(), we only generate trips and config
generator.generate_trips()
generator.generate_config()

paths = generator.paths
print(f"Scenario configured using: {paths.config_file}")


print(f"Simulation Horizon: {SIMULATION_DURATION}s | Traffic Spawning Until: {TRAFFIC_SPAWN_DURATION}s")
print("Traffic-light agents:", custom_tls_ids)


Found OSM network file 'lancaster_373m_35.osm.xml', converting to 'lancaster_373m_35.net.xml'...
Success.
Conversion complete. Network file 'lancaster_373m_35.net.xml' created at /content/marl-tsc/outputs/lancaster_373m_35.net.xml
Network file 'lancaster_373m_35.net.xml' already exists at /content/marl-tsc/outputs/lancaster_373m_35.net.xml. No move needed.
Scenario configured using: /content/marl-tsc/outputs/config.sumocfg
Simulation Horizon: 2000000s | Traffic Spawning Until: 2400000s
Traffic-light agents: ['5464450922', 'cluster_1111602959_6308330_6398665769_6398666101', 'cluster_1112902336_2338735_6300917286', 'cluster_1395459938_3904465077_6288784963', 'cluster_1671362532_2217405_253943174_3904937313_#3more', 'cluster_2217373_5464742560_610595545_6288784968_#6more', 'cluster_27212021_5464547758_6299153326', 'joinedS_3903073405_cluster_5835265011_610595547', 'joinedS_6528610881_cluster_1305014216_5464547754_6100910_6528610880_cluster_2217469_5464547745_5464547748_5464547749', 'joine

In [52]:
import sumolib
net = sumolib.net.readNet(str(paths.network_file))
tls_in_net = [tls.getID() for tls in net.getTrafficLights()]
print(f"Traffic lights found in net file: {len(tls_in_net)}")
print(tls_in_net)

Traffic lights found in net file: 11
['cluster_1395459938_3904465077_6288784963', 'joinedS_6528610881_cluster_1305014216_5464547754_6100910_6528610880_cluster_2217469_5464547745_5464547748_5464547749', 'cluster_1670999797_1670999799_1670999973_2217366_#2more', 'cluster_1111602959_6308330_6398665769_6398666101', 'joinedS_2217373_6288784968_6288784969_6288784970_#2more', 'cluster_27212021_5464547758_6299153326', 'joinedS_cluster_3903235205_3903235208_6288785520_cluster_3903235206_3903235211_3903235213_6288785518_#1more_cluster_5178274273_6288785516', 'joinedS_3903073405_cluster_5835265011_610595547', 'cluster_1112902336_2338735_6300917286', 'cluster_1671362532_2217405_253943174_3904937313_#3more', '5464450922']


## 2. Run the simple baselines

Random and fixed-time are useful sanity checks before training. They use the same environment and the same evaluation seeds as the learned policies.


In [54]:
import traci
import sys

def reset_traci_connection():
    # Force clear TraCI's internal connection registries
    try:
        traci.switch("default")
        traci.close()
    except Exception:
        pass # No active connection to close

    try:
        # Completely purge the connection dictionary tracking connections
        if 'traci.main' in sys.modules:
            traci.main._connections.clear()
            print("Purged TraCI connections dictionary.")
    except Exception:
        pass

    # Reset the module completely out of Python sys memory
    if 'traci' in sys.modules:
        del sys.modules['traci']
        # Re-import fresh only if needed by subsequent code, otherwise it's handled by SumoTrafficEnv
    # print("TraCI state successfully reset!")


# ==========================================
# 1. RANDOM BASELINE EVALUATION
# ==========================================
reset_traci_connection()
random_result = evaluate_policy(
    config_file=paths.config_file,
    traffic_light_ids=None,             # Dynamically extract from the net file
    policy=random_actions,
    episodes=1,#EVALUATION_EPISODES,       # Use 10 episodes to match MAPPO evaluation
    max_steps=600, #EPISODE_STEPS,            # FIX: Must be 600 steps, not 3000 duration
    seed=42,
    env_kwargs={
        "green_phase_count": 8, # Changed from 4 to 8 to match discovered traffic light phases
        "min_green_seconds": MIN_GREEN_SECONDS,
        "seconds_per_action": SECONDS_PER_ACTION,
        #"max_steps": 600,     # Pass to ensure internal alignment
    },
)

# ==========================================
# 2. FIXED-TIME BASELINE EVALUATION
# ==========================================
reset_traci_connection()
fixed_time_result = evaluate_policy(
    config_file=paths.config_file,
    traffic_light_ids=None,             # Dynamically extract from the net file
    policy=fixed_time_actions,
    episodes= 1,#EVALUATION_EPISODES,       # Use 10 episodes to match MAPPO evaluation
    max_steps= 600, #EPISODE_STEPS,            # FIX: Must be 600 steps, not 3000 duration
    seed=42,
    env_kwargs={
        "green_phase_count": 8, # Changed from 4 to 8 to match discovered traffic light phases
        "min_green_seconds": MIN_GREEN_SECONDS,
        "seconds_per_action": SECONDS_PER_ACTION,
        #"max_steps": 600,     # Pass to ensure internal alignment
    },
)

 Retrying in 1 seconds
 Retrying in 1 seconds
Actions: {'5464450922': 0, 'cluster_1111602959_6308330_6398665769_6398666101': 0, 'cluster_1112902336_2338735_6300917286': 0, 'cluster_1395459938_3904465077_6288784963': 0, 'cluster_1670999797_1670999799_1670999973_2217366_#2more': 0, 'cluster_1671362532_2217405_253943174_3904937313_#3more': 0, 'cluster_27212021_5464547758_6299153326': 0, 'joinedS_2217373_6288784968_6288784969_6288784970_#2more': 0, 'joinedS_3903073405_cluster_5835265011_610595547': 0, 'joinedS_6528610881_cluster_1305014216_5464547754_6100910_6528610880_cluster_2217469_5464547745_5464547748_5464547749': 0, 'joinedS_cluster_3903235205_3903235208_6288785520_cluster_3903235206_3903235211_3903235213_6288785518_#1more_cluster_5178274273_6288785516': 0}
Actions: {'5464450922': 0, 'cluster_1111602959_6308330_6398665769_6398666101': 0, 'cluster_1112902336_2338735_6300917286': 0, 'cluster_1395459938_3904465077_6288784963': 0, 'cluster_1670999797_1670999799_1670999973_2217366_#2more'

In [55]:
baseline_rows = [
    {
        "Policy": "Random",
        "Episodes": random_result["episodes"],
        "Mean total reward": random_result["mean_total_reward"],
        "Mean local queue": random_result["mean_local_queue"],
        "Total completed steps": random_result["total_completed_steps"],
        "Mean switches / episode": random_result["mean_switches_per_episode"],
    },
    {
        "Policy": "Fixed-Time",
        "Episodes": fixed_time_result["episodes"],
        "Mean total reward": fixed_time_result["mean_total_reward"],
        "Mean local queue": fixed_time_result["mean_local_queue"],
        "Total completed steps": fixed_time_result["total_completed_steps"],
        "Mean switches / episode": fixed_time_result["mean_switches_per_episode"],
    },
]
print_comparison_table(baseline_rows)

Policy     | Episodes | Mean total reward | Mean local queue | Total completed steps | Mean switches / episode
-----------+----------+-------------------+------------------+-----------------------+------------------------
Random     | 1        | -388.744          | 0.348            | 600                   | 1532.000               
Fixed-Time | 1        | -689.274          | 0.772            | 600                   | 1386.000               


## 3. Train

PPO uses parameter sharing through the PettingZoo/SuperSuit/SB3 wrapper. MAPPO adds a centralized critic and a shared decentralized actor.


### PPO

In [59]:
ppo_model, ppo_history, ppo_model_path = train_ppo(
    config_file=paths.config_file,
    traffic_light_ids=None,
    output_dir=OUTPUT_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    max_steps=SIMULATION_DURATION,
    seed=42,
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

 Retrying in 1 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


IndexError: list index out of range

In [ ]:

fig, ax = plot_training_histories(ppo_history)
plt.show()

print("Saved PPO model:", ppo_model_path)

### MAPPO

In [64]:
mappo_model, mappo_history, mappo_model_path = train_mappo(
    config_file=paths.config_file,
    traffic_light_ids=None,
    output_dir=OUTPUT_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    rollout_steps=1024,
    max_steps=SIMULATION_DURATION,
    seed=42,
    env_kwargs={"green_phase_count": 8,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

fig, ax = plot_training_histories(mappo_history)
plt.show()

print("Saved MAPPO model:", mappo_model_path)

 Retrying in 1 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


 Retrying in 1 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[MAPPO] Episode 1 | Return: -45.08 | Total Steps: 1032
 Retrying in 1 seconds


...

[MAPPO] Episode 736 | Return: -34.93 | Total Steps: 750592
 Retrying in 1 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[MAPPO] Episode 737 | Return: -26.82 | Total Steps: 751678
 Retrying in 1 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


KeyboardInterrupt: 

## 4. Evaluate all policies on the same seeds

The final comparison table keeps the baseline runs and adds the shared PPO and MAPPO policies on the same evaluation seeds.


In [ ]:
shared_mappo_result = evaluate_policy(
    config_file=paths.config_file,
    # 1. Pass None to dynamically extract the agents from the network file, matching training!
    traffic_light_ids=traffic_light_ids,
    policy=mappo_model,
    episodes=1, #EVALUATION_EPISODES,
    # 2. FIX: Must be EPISODE_STEPS, not SIMULATION_DURATION
    max_steps=EPISODE_STEPS,
    seed=42,
    # 3. Pass every environment hyperparameter to ensure observation spaces align perfectly
    env_kwargs={
        "green_phase_count": 4,
        "min_green_seconds": MIN_GREEN_SECONDS,
        "seconds_per_action": SECONDS_PER_ACTION,
        #"max_steps": EPISODE_STEPS,
    },
)

In [ ]:
shared_ppo_result = evaluate_policy(
    config_file=paths.config_file,
    # 1. Pass None to dynamically extract the agents from the network file, matching training!
    traffic_light_ids=traffic_light_ids,
    policy=ppo_model,
    episodes=1, #EVALUATION_EPISODES,
    # 2. FIX: Must be EPISODE_STEPS, not SIMULATION_DURATION
    max_steps=EPISODE_STEPS,
    seed=42,
    # 3. Pass every environment hyperparameter to ensure observation spaces align perfectly
    env_kwargs={
        "green_phase_count": 4,
        "min_green_seconds": MIN_GREEN_SECONDS,
        "seconds_per_action": SECONDS_PER_ACTION,
        #"max_steps": EPISODE_STEPS,
    },
)

In [ ]:
comparison_rows = baseline_rows + [
    {
        "Policy": "Shared PPO",
        "Episodes": shared_ppo_result["episodes"],
        "Mean total reward": shared_ppo_result["mean_total_reward"],
        "Mean local queue": shared_ppo_result["mean_local_queue"],
        "Total completed steps": shared_ppo_result["total_completed_steps"],
        "Mean switches / episode": shared_ppo_result["mean_switches_per_episode"],
    },
#comparison_rows = baseline_rows + [
        {
        "Policy": "MAPPO",
        "Episodes": shared_mappo_result["episodes"],
        "Mean total reward": shared_mappo_result["mean_total_reward"],
        "Mean local queue": shared_mappo_result["mean_local_queue"],
        "Total completed steps": shared_mappo_result["total_completed_steps"],
        "Mean switches / episode": shared_mappo_result["mean_switches_per_episode"],
    },
]

print_comparison_table(comparison_rows)


### Traci Escape Hatch

In [63]:
on = True
if on:
    # Stop Traci processes if not already stopped, to prevent connection issues in subsequent runs
    import sys

    # 1. Force clear TraCI's internal connection registries
    try:
        import traci
        traci.switch("default")
        traci.close()
    except Exception:
        pass

    try:
        # Completely purge the connection dictionary tracking connections
        import traci.main
        traci.main._connections.clear()
        print("Purged TraCI connections dictionary.")
    except Exception:
        pass

    # 2. Reset the module completely out of Python sys memory
    if 'traci' in sys.modules:
        del sys.modules['traci']
        print("Flushed TraCI module from Python runtime memory.")

    # 3. Re-import fresh
    import traci
    print("TraCI state successfully reset!")

Flushed TraCI module from Python runtime memory.
TraCI state successfully reset!


# Graph Stuff


In [ ]:
generator_graphs = SimulationGenerator(
    output_dir=OUTPUT_DIR,
    grid_number=4,
    lane_number=2,
    traffic_light_ids=traffic_light_ids,
    trip_begin=0,
    trip_end=2_000,#TRAFFIC_SPAWN_DURATION,  # TEMPORARY TEST
    trip_period=2,                    # Steady stream: 1 car every 2 seconds
    seed=42,
)

In [ ]:
#paths = generator_graphs.generate_all()
paths = generator.generate_all()

### Environment and Smoke Tests

In [ ]:
from marl_tsc.graph_based.graph_builder import GraphBuilder

builder = GraphBuilder(paths.network_file)

topology = builder.build()
print("Topology output:")
print("List of Nodes")
print(topology.agent_ids)
print("Edge Index:Columns Represent source --> Target")
print(topology.edge_index)

In [ ]:
tls = next(iter(builder.net.getTrafficLights()))

connections = tls.getConnections()

print(connections[0])

incoming_lane = connections[0][0]
outgoing_lane = connections[0][1]

print(type(incoming_lane))
print(type(outgoing_lane))

print(sorted(
    attr
    for attr in dir(incoming_lane)
    if not attr.startswith("_")
)[:50])

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()

for agent in topology.agent_ids:
    G.add_node(agent)

for src, dst in topology.edge_index.t().tolist():
    G.add_edge(
        topology.agent_ids[src],
        topology.agent_ids[dst]
    )

plt.figure(figsize=(6, 6))

nx.draw(
    G,
    with_labels=True,
    node_size=2000,
)

plt.show()

In [ ]:
from marl_tsc.graph_based.graph_builder import GraphBuilder
from marl_tsc.graph_based.graph_env import GraphTrafficEnv

from marl_tsc.graph_based.encoders.gat_encoder import GATEncoder

from marl_tsc.graph_based.models.actor import ActorHead
from marl_tsc.graph_based.models.critic import CriticHead
from marl_tsc.graph_based.models.graph_policy import GraphPolicy


# ------------------------------------------------------------------
# 1. Build graph topology
# ------------------------------------------------------------------

topology = GraphBuilder(
    paths.network_file
).build()

print("Agents:", topology.agent_ids)
print("Edge Index Shape:", topology.edge_index.shape)

assert len(topology.agent_ids) > 0
assert topology.edge_index.shape[0] == 2


# ------------------------------------------------------------------
# 2. Create graph environment
# ------------------------------------------------------------------

env = GraphTrafficEnv(
    config_file=paths.config_file,
    network_file=paths.network_file,
    possible_agents=topology.agent_ids,
    max_steps=100,
    seed=42,
)

graph_obs, infos = env.reset(seed=42)

print("\nGraph:")
print(graph_obs)

assert graph_obs.graph.x.shape[0] == len(topology.agent_ids)
assert graph_obs.graph.edge_index.shape[0] == 2


# ------------------------------------------------------------------
# 3. Encoder
# ------------------------------------------------------------------

obs_dim = graph_obs.graph.x.shape[1]

encoder = GATEncoder(
    obs_dim=obs_dim,
    hidden_dim=64,
)

encoder_output = encoder(graph_obs)

print("\nEncoder:")
print("Node Embeddings:", encoder_output.node_embeddings.shape)
print("Graph Embedding:", encoder_output.graph_embedding.shape)

assert encoder_output.node_embeddings.shape == (
    len(topology.agent_ids),
    64,
)

assert encoder_output.graph_embedding.shape == (64,)


# ------------------------------------------------------------------
# 4. Heads
# ------------------------------------------------------------------

action_dim = env.action_spaces[
    topology.agent_ids[0]
].n

actor = ActorHead(
    embedding_dim=64,
    action_dim=action_dim,
)

critic = CriticHead(
    embedding_dim=64,
)

logits = actor(encoder_output)
value = critic(encoder_output)

print("\nHeads:")
print("Logits:", logits.shape)
print("Value:", value.shape)

assert logits.shape == (
    len(topology.agent_ids),
    action_dim,
)

assert value.shape == (1,)


# ------------------------------------------------------------------
# 5. Full policy
# ------------------------------------------------------------------

policy = GraphPolicy(
    encoder=encoder,
    actor_head=actor,
    critic_head=critic,
)

output = policy(graph_obs)

print("\nPolicy:")
print("Policy Logits:", output.logits.shape)
print("Policy Value:", output.value.shape)

assert output.logits.shape == (
    len(topology.agent_ids),
    action_dim,
)

assert output.value.shape == (1,)

print("\nGRAPH STACK SMOKE TEST PASSED")

env.close()

In [ ]:
import torch
import torch_geometric
from marl_tsc.graph_based.graph_runner import GraphRunner
from marl_tsc.graph_based.graph_rollout import GraphRollout
from marl_tsc.graph_based.advantage_estimator import (
    AdvantageEstimator,
)

# --------------------------------------------------
# 1. Collect rollout
# --------------------------------------------------

runner = GraphRunner(
    env=env,
    policy=policy,
)

rollout = runner.collect_rollout(
    num_steps=10,
    seed=42,
)

print("Transitions:", len(rollout))

assert len(rollout) > 0

# --------------------------------------------------
# 2. Transition checks
# --------------------------------------------------

first = rollout[0]

print("\nTransition:")
print(type(first))

print("Actions:", first.action_dict)
print("Log Prob Shape:", first.log_prob.shape)
print("Logits Shape:", first.logits.shape)
print("Value Shape:", first.value.shape)

assert first.log_prob.shape == (
    len(topology.agent_ids),
)

assert first.logits.shape[0] == len(
    topology.agent_ids
)

assert first.value.shape == (1,)

# --------------------------------------------------
# 3. Rollout batch
# --------------------------------------------------

batch = GraphRollout.from_transitions(
    rollout,
    topology.agent_ids,
)

print("\nRollout Batch:")
print("Actions:", batch.actions.shape)
print("Log Probs:", batch.log_probs.shape)
print("Logits:", batch.logits.shape)
print("Values:", batch.values.shape)
print("Rewards:", batch.rewards.shape)
print("Dones:", batch.dones.shape)

assert batch.actions.shape == (
    len(rollout),
    len(topology.agent_ids),
)

assert batch.log_probs.shape == (
    len(rollout),
    len(topology.agent_ids),
)

assert batch.rewards.shape == (
    len(rollout),
    len(topology.agent_ids),
)

assert batch.values.shape == (
    len(rollout),
)

assert batch.dones.shape == (
    len(rollout),
)

# --------------------------------------------------
# 4. Advantage estimation
# --------------------------------------------------

adv_batch = (
    AdvantageEstimator
    .compute_gae(batch)
)

print("\nAdvantages:")
print(
    "Advantages:",
    adv_batch.advantages.shape,
)

print(
    "Returns:",
    adv_batch.returns.shape,
)

assert (
    adv_batch.advantages.shape
    ==
    batch.values.shape
)

assert (
    adv_batch.returns.shape
    ==
    batch.values.shape
)

assert not torch.isnan(
    adv_batch.advantages
).any()

assert not torch.isnan(
    adv_batch.returns
).any()

# --------------------------------------------------
# 5. Sample outputs
# --------------------------------------------------

print("\nSample Values")

print(
    "Advantages:",
    adv_batch.advantages[:5]
)

print(
    "Returns:",
    adv_batch.returns[:5]
)

print(
    "\n✅ PHASE 2 GRAPH ROLLOUT PIPELINE PASSED"
)

In [ ]:
def evaluate_policy(env, policy, steps=100):

    graph_obs, infos = env.reset()

    total_reward = 0

    for _ in range(steps):

        output = policy(graph_obs)

        actions = output.logits.argmax(
            dim=1
        )

        action_dict = {
            agent_id: int(action)
            for agent_id, action in zip(
                graph_obs.agent_ids,
                actions,
            )
        }

        (
            graph_obs,
            rewards,
            terms,
            truncs,
            infos,
        ) = env.step(action_dict)

        total_reward += sum(
            rewards.values()
        )

    return total_reward

In [ ]:
num_params = sum(
    p.numel()
    for p in policy.parameters()
)

print(
    f"Trainable parameters: {num_params:,}"
)

In [ ]:
import copy

optimizer = torch.optim.Adam(
    policy.parameters(),
    lr=2e-4,
)

from marl_tsc.graph_based.graph_ctde_trainer import (
    GraphCTDETrainer,
)

trainer = GraphCTDETrainer(
    env=env,
    policy=policy,
    optimizer=optimizer,
    rollout_steps=10,
)

before = copy.deepcopy(
    next(
        policy.parameters()
    ).detach()
)

stats = trainer.train_step()

after = next(
    policy.parameters()
).detach()

print(stats)

print(
    torch.equal(
        before,
        after,
    )
)

In [ ]:
import importlib

import marl_tsc.graph_based.graph_runner

importlib.reload(
    #marl_tsc.graph_based.graph_runner,
    marl_tsc.graph_based.graph_rollout
)

from marl_tsc.graph_based.graph_runner import GraphRunner
import marl_tsc.graph_based.graph_rollout

In [ ]:
import torch

# ----------------------------------
# Evaluation helper
# ----------------------------------

def evaluate_policy(env, policy, steps=100):

    graph_obs, _ = env.reset()

    total_reward = 0.0

    for _ in range(steps):

        with torch.no_grad():

            output = policy(graph_obs)

            actions = output.logits.argmax(dim=1)

        action_dict = {
            agent_id: int(action)
            for agent_id, action in zip(
                graph_obs.agent_ids,
                actions,
            )
        }

        (
            graph_obs,
            rewards,
            terminations,
            truncations,
            infos,
        ) = env.step(action_dict)

        total_reward += sum(
            rewards.values()
        )

        if any(truncations.values()):
            break

    return total_reward


# ----------------------------------
# Fresh optimizer
# ----------------------------------

optimizer = torch.optim.Adam(
    policy.parameters(),
    lr=1e-3,
)

# ----------------------------------
# Trainer
# ----------------------------------

trainer = GraphCTDETrainer(
    env=env,
    policy=policy,
    optimizer=optimizer,
    rollout_steps=64,
)

# ----------------------------------
# Baseline evaluation
# ----------------------------------

before_reward = evaluate_policy(
    env,
    policy,
)

print(f"Before Training: {before_reward:.4f}")

# ----------------------------------
# Train
# ----------------------------------

for iteration in range(100):

    stats = trainer.train_step()

    if (iteration + 1) % 10 == 0:

        print(
            f"Iter {iteration + 1:3d} | "
            f"Actor {stats['actor_loss']:.4f} | "
            f"Critic {stats['critic_loss']:.4f}"
        )

# ----------------------------------
# Final evaluation
# ----------------------------------

after_reward = evaluate_policy(
    env,
    policy,
)

print()
print(f"After Training:  {after_reward:.4f}")
print(f"Improvement:     {after_reward - before_reward:.4f}")

# ----------------------------------
# Inspect learned policy
# ----------------------------------

graph_obs, _ = env.reset()

with torch.no_grad():

    output = policy(graph_obs)

print()
print("Greedy Actions:")
print(output.logits.argmax(dim=1))

print()
print("Logits:")
print(output.logits)

In [ ]:
def eval_constant(env, action, steps=100):

    graph_obs, _ = env.reset()

    total_reward = 0

    for _ in range(steps):

        action_dict = {
            agent: action
            for agent in env.agent_ids
        }

        graph_obs, rewards, *_ = env.step(
            action_dict
        )

        total_reward += sum(
            rewards.values()
        )

    return total_reward


print(eval_constant(env, 0))
print(eval_constant(env, 1))
print(eval_constant(env, 2))
print(eval_constant(env, 3))

In [ ]:
from marl_tsc.graph_based.base_trainer import BaseGraphTrainer


class DummyTrainer(BaseGraphTrainer):

    def update(
        self,
        rollout_batch,
        advantage_batch,
    ):

        print(type(rollout_batch))
        print(type(advantage_batch))

        print()

        print("Actions:")
        print(rollout_batch.actions.shape)

        print()

        print("Advantages:")
        print(advantage_batch.advantages.shape)

        print()

        print("Returns:")
        print(advantage_batch.returns.shape)

        return {
            "dummy_loss": 0.0,
        }


trainer = DummyTrainer(
    env=env,
    policy=policy,
    optimizer=optimizer,
    rollout_steps=10,
)

stats = trainer.train_step()

print()
print(stats)

In [57]:
from marl_tsc.graph_based.train_graph_ctde import (
    train_graph_ctde,
)

model, history, model_path = (
    train_graph_ctde(
        config_file=paths.config_file,
        network_file=paths.network_file,
        traffic_light_ids=traffic_light_ids,
        output_dir=OUTPUT_DIR,
        total_timesteps=1000,
        rollout_steps=64,
        env_kwargs={
            "green_phase_count": GREEN_PHASE_COUNT,
            "min_green_seconds": MIN_GREEN_SECONDS,
        },
    )
)

print(type(model))
print(len(history))
print(model_path)
print(history[-1])

/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


SyntaxError: invalid syntax. Perhaps you forgot a comma? (graph_ctde_trainer.py, line 74)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
fig, ax = plot_training_histories(history)
plt.show()

#### Git stuff

In [ ]:
!git status


In [ ]:
!git add .

In [ ]:
import shlex
commit_msg = '''
Bug within GraphCTDETrainer. Needs commiting before reloading notebook
'''
'''
TODO:
- Heterogenous networks
- Full Training run on bigger networks
- Possible integration with Andy's updated code
- Possible integration with Tom's networks
'''

In [ ]:
print(commit_msg)

In [ ]:
!git config --global user.email "zakfayle@gmail.com"
!git config --global user.name "IsaacFayle-Waters"
!git commit -m {shlex.quote(commit_msg)}

In [ ]:
!git push origin zak_new